# GPDK CPW wave-port mesh regression

Reproduce [issue #262](https://github.com/gdsfactory/gsim/issues/262) and check the fix from [PR #267](https://github.com/gdsfactory/gsim/pull/267). The affected case uses the generic GDSFactory PDK, the `M1` layout layer, and full-size `metal1` wave ports.

Install this checkout with `uv sync --dev` and choose its `.venv/bin/python` kernel. Run through **Check wave-port topology** for a local test requiring no Palace installation or cloud account. Every wave-port triangle must be attached to exactly one tetrahedron. The old implementation created orphan triangles below the airbox and Palace aborted before solving.

The final section runs a three-frequency cloud simulation. For a local Palace installation, replace that run with `sim.run_local(palace_executable="/path/to/palace")`.

## Define the GPDK CPW

In [ ]:
import gdsfactory as gf
from gdsfactory.gpdk import LAYER

gf.gpdk.PDK.activate()


@gf.cell
def gsg_electrode(
    length: float = 800,
    s_width: float = 20,
    g_width: float = 40,
    gap_width: float = 15,
    layer=LAYER.M1,
) -> gf.Component:
    """
    Create a GSG (Ground-Signal-Ground) electrode.

    Args:
        length: horizontal length of the electrodes
        s_width: width of the signal (center) electrode
        g_width: width of the ground electrodes
        gap_width: gap between signal and ground electrodes
        layer: layer for the metal
    """
    c = gf.Component()

    r1 = c << gf.c.rectangle((length, g_width), centered=True, layer=layer)
    r1.move((0, (g_width + s_width) / 2 + gap_width))

    _r2 = c << gf.c.rectangle((length, s_width), centered=True, layer=layer)

    r3 = c << gf.c.rectangle((length, g_width), centered=True, layer=layer)
    r3.move((0, -(g_width + s_width) / 2 - gap_width))

    c.add_port(
        name="o1",
        center=(-length / 2, 0),
        width=s_width,
        orientation=180,
        port_type="electrical",
        layer=layer,
    )

    c.add_port(
        name="o2",
        center=(length / 2, 0),
        width=s_width,
        orientation=0,
        port_type="electrical",
        layer=layer,
    )

    return c


c = gsg_electrode()
cc = c.copy()
cc.draw_ports()
cc

## Configure and mesh

In [ ]:
from gsim.common.stack import get_stack
from gsim.palace import DrivenSim

sim = DrivenSim()
sim.set_output_dir("./palace-sim-gpdk-waveport-smoke")
sim.set_geometry(c)
sim.set_stack(get_stack())
sim.set_airbox(margin_x=0.0, margin_y=50.0, z_above=100.0, z_below=100.0)
for port_name, excited in [("o1", True), ("o2", False)]:
    sim.add_wave_port(port_name, layer="metal1", max_size=True, mode=1, excited=excited)
sim.set_driven(fmin=1e9, fmax=100e9, num_points=3)
validation = sim.validate_config()
assert validation.valid, validation.errors

mesh_result = sim.mesh(
    preset="default", refined_mesh_size=2.0, max_mesh_size=40.0, fmax=150e9
)
config_path = sim.write_config()

## Check wave-port topology

Inspect the generated mesh, independently of the domain-bound calculations. The expected result is **zero orphan wave-port faces** and one adjacent tetrahedron for each port triangle.

In [ ]:
import json
from collections import Counter

import meshio
import numpy as np

config = json.loads(config_path.read_text())
port_attributes = {
    attribute
    for port in config["Boundaries"]["WavePort"]
    for attribute in port["Attributes"]
}
assert len(config["Boundaries"]["WavePort"]) == 2
mesh = meshio.read(mesh_result.mesh_path)
tetrahedra = mesh.cells_dict["tetra"]
triangles = mesh.cells_dict["triangle"]
triangle_attributes = mesh.cell_data_dict["gmsh:physical"]["triangle"]
face_corners = [(0, 1, 2), (0, 1, 3), (0, 2, 3), (1, 2, 3)]
volume_face_counts = Counter(
    tuple(sorted(tetrahedron[list(corners)]))
    for tetrahedron in tetrahedra
    for corners in face_corners
)

for attribute in sorted(port_attributes):
    port_triangles = triangles[triangle_attributes == attribute]
    assert len(port_triangles) > 0, f"Port attribute {attribute} has no triangles"
    adjacent_tetrahedra = np.array(
        [volume_face_counts[tuple(sorted(face))] for face in port_triangles]
    )
    orphan_count = np.count_nonzero(adjacent_tetrahedra == 0)
    port_z = mesh.points[np.unique(port_triangles), 2]
    print(
        f"Port attribute {attribute}: {len(port_triangles)} triangles, "
        f"{orphan_count} orphan faces, z=[{port_z.min():.3f}, {port_z.max():.3f}] um"
    )
    assert np.all(adjacent_tetrahedra == 1), (
        f"Port attribute {attribute} has triangles without exactly one adjacent tetrahedron"
    )
print("Wave-port topology checks passed.")

## Run Palace (cloud)

This cell requires a GDSFactory+ account and submits a new job. With the fix, the mesh should pass preprocessing and produce two-port S-parameters at three frequencies. Cache reuse is disabled so the mesh is exercised.

In [ ]:
results = sim.run(check_cache=False, verbose="full")
results.plot_interactive()